In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.impute import SimpleImputer
import torch
from torch import nn
import copy





In [2]:
PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "data" / "node_features.csv").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Could not find data/node_features.csv.")
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "node_features.csv"



In [3]:
df = pd.read_csv(DATA_PATH)

feature_columns = [
    column
    for column in df.columns
    if column not in {"username", "target_rating"}
]

X = df[feature_columns]
y = df["target_rating"]

# print("Input features:", feature_columns)
# print("X shape:", X.shape)
# print("y shape:", y.shape)
# print("Non-numeric features:", X.select_dtypes(exclude="number").columns.tolist())
# print("Total missing feature values:", X.isna().sum().sum())



In [4]:
RANDOM_SEED = 42

train_df, remaining_df = train_test_split(
    df,
    test_size=0.30,
    random_state=RANDOM_SEED,
)

val_df, test_df = train_test_split(
    remaining_df,
    test_size=0.50,
    random_state=RANDOM_SEED,
)

X_train = train_df[feature_columns]
y_train = train_df["target_rating"]

X_val = val_df[feature_columns]
y_val = val_df["target_rating"]

X_test = test_df[feature_columns]
y_test = test_df["target_rating"]

train_users = set(train_df["username"])
val_users = set(val_df["username"])
test_users = set(test_df["username"])

assert train_users.isdisjoint(val_users)
assert train_users.isdisjoint(test_users)
assert val_users.isdisjoint(test_users)

# print("Training samples:", len(train_df))
# print("Validation samples:", len(val_df))
# print("Test samples:", len(test_df))
# print("Player overlap check passed.")



In [5]:
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val = X_val.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

feature_imputer = SimpleImputer(strategy="median")

X_train_imputed = feature_imputer.fit_transform(X_train)
X_val_imputed = feature_imputer.transform(X_val)
X_test_imputed = feature_imputer.transform(X_test)

feature_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train_imputed)
X_val_scaled = feature_scaler.transform(X_val_imputed)
X_test_scaled = feature_scaler.transform(X_test_imputed)

assert np.isfinite(X_train_scaled).all()
assert np.isfinite(X_val_scaled).all()
assert np.isfinite(X_test_scaled).all()

# print("Scaled training shape:", X_train_scaled.shape)
# print("Training feature means:", X_train_scaled.mean(axis=0).round(3))
# print(
#     "Training feature standard deviations:",
#     X_train_scaled.std(axis=0).round(3),
# )
# print("All scaled feature values are finite.")



In [6]:
target_scaler = StandardScaler()

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_scaled = target_scaler.fit_transform(
    y_train.to_numpy().reshape(-1, 1)
)
y_val_scaled = target_scaler.transform(
    y_val.to_numpy().reshape(-1, 1)
)
y_test_scaled = target_scaler.transform(
    y_test.to_numpy().reshape(-1, 1)
)

y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

# print("Training target mean:", y_train_tensor.mean().item())
# print(
#     "Training target standard deviation:",
#     y_train_tensor.std(unbiased=False).item(),
# )

# print("Training input shape:", X_train_tensor.shape)
# print("Training target shape:", y_train_tensor.shape)
# print("Input dtype:", X_train_tensor.dtype)



In [7]:
torch.manual_seed(RANDOM_SEED)

model = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 64),
    nn.ReLU(),
    nn.Dropout(p=0.1),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(p=0.1),
    nn.Linear(32, 1),
)

model.eval()

with torch.no_grad():
    sample_predictions = model(X_train_tensor[:5])

# print(model)
# print("Sample prediction shape:", sample_predictions.shape)



In [8]:
############### model parameters update and start trainning
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

NUM_EPOCHS = 200
PATIENCE = 20

best_val_loss = float("inf")
best_epoch = 0
best_model_state = None
epochs_without_improvement = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    train_predictions = model(X_train_tensor)
    train_loss = loss_function(train_predictions, y_train_tensor)

    train_loss.backward()
    optimizer.step()

    model.eval()

    with torch.no_grad():
        val_predictions = model(X_val_tensor)
        val_loss = loss_function(val_predictions, y_val_tensor)

    current_val_loss = val_loss.item()

    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss
        best_epoch = epoch
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Train MSE: {train_loss.item():.4f} | "
            f"Validation MSE: {current_val_loss:.4f}"
        )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

model.load_state_dict(best_model_state)

print(f"Best epoch: {best_epoch}")
print(f"Best validation MSE: {best_val_loss:.4f}")



Epoch 001 | Train MSE: 1.0009 | Validation MSE: 0.9103
Epoch 010 | Train MSE: 0.7315 | Validation MSE: 0.6531
Epoch 020 | Train MSE: 0.5265 | Validation MSE: 0.4709
Epoch 030 | Train MSE: 0.4378 | Validation MSE: 0.4099
Epoch 040 | Train MSE: 0.4219 | Validation MSE: 0.3893
Epoch 050 | Train MSE: 0.4050 | Validation MSE: 0.3712
Epoch 060 | Train MSE: 0.3957 | Validation MSE: 0.3685
Epoch 070 | Train MSE: 0.3957 | Validation MSE: 0.3646
Epoch 080 | Train MSE: 0.3838 | Validation MSE: 0.3630
Epoch 090 | Train MSE: 0.3860 | Validation MSE: 0.3608
Epoch 100 | Train MSE: 0.3829 | Validation MSE: 0.3602
Epoch 110 | Train MSE: 0.3773 | Validation MSE: 0.3585
Epoch 120 | Train MSE: 0.3798 | Validation MSE: 0.3577
Epoch 130 | Train MSE: 0.3733 | Validation MSE: 0.3561
Epoch 140 | Train MSE: 0.3707 | Validation MSE: 0.3553
Epoch 150 | Train MSE: 0.3754 | Validation MSE: 0.3542
Epoch 160 | Train MSE: 0.3648 | Validation MSE: 0.3534
Epoch 170 | Train MSE: 0.3654 | Validation MSE: 0.3522
Epoch 180 

In [9]:
model.eval()

with torch.no_grad():
    val_predictions_scaled = model(X_val_tensor).cpu().numpy()

val_predictions_rating = target_scaler.inverse_transform(
    val_predictions_scaled
).ravel()

val_targets_rating = y_val.to_numpy()

val_errors = val_predictions_rating - val_targets_rating

val_mae = np.mean(np.abs(val_errors))
val_rmse = np.sqrt(np.mean(val_errors ** 2))

print(f"Validation MAE: {val_mae:.2f} rating points")
print(f"Validation RMSE: {val_rmse:.2f} rating points")



Validation MAE: 145.89 rating points
Validation RMSE: 184.18 rating points


In [10]:
mean_rating = y_train.mean()

mean_predictions = np.full(
    shape=len(y_val),
    fill_value=mean_rating,
)

mean_errors = mean_predictions - val_targets_rating

mean_mae = np.mean(np.abs(mean_errors))
mean_rmse = np.sqrt(np.mean(mean_errors ** 2))

print(f"Training mean rating: {mean_rating:.2f}")
print(f"Mean baseline validation MAE: {mean_mae:.2f} rating points")
print(f"Mean baseline validation RMSE: {mean_rmse:.2f} rating points")

rmse_reduction = (mean_rmse - val_rmse) / mean_rmse * 100

print(f"MLP validation RMSE reduction: {rmse_reduction:.2f}%")

Training mean rating: 1531.82
Mean baseline validation MAE: 240.20 rating points
Mean baseline validation RMSE: 303.90 rating points
MLP validation RMSE reduction: 39.39%


In [11]:
from sklearn.linear_model import Ridge

ridge_model = Ridge(alpha=1.0)

ridge_model.fit(
    X_train_scaled,
    y_train.to_numpy(),
)

ridge_val_predictions = ridge_model.predict(X_val_scaled)
ridge_val_errors = ridge_val_predictions - y_val.to_numpy()

ridge_val_mae = np.mean(np.abs(ridge_val_errors))
ridge_val_rmse = np.sqrt(np.mean(ridge_val_errors ** 2))

print(f"Ridge validation MAE: {ridge_val_mae:.2f} rating points")
print(f"Ridge validation RMSE: {ridge_val_rmse:.2f} rating points")

Ridge validation MAE: 153.12 rating points
Ridge validation RMSE: 193.83 rating points


In [12]:
comparison = pd.DataFrame(
    [
        {
            "Model": "Training mean",
            "Validation MAE": mean_mae,
            "Validation RMSE": mean_rmse,
        },
        {
            "Model": "Ridge",
            "Validation MAE": ridge_val_mae,
            "Validation RMSE": ridge_val_rmse,
        },
        {
            "Model": "MLP",
            "Validation MAE": val_mae,
            "Validation RMSE": val_rmse,
        },
    ]
)

comparison

In [13]:
# Repeat training with fixed player splits and different training seeds.
TRAINING_SEEDS = [42, 43, 44]
seed_results = []
seed_validation_predictions = {}
seed_model_states = {}

for training_seed in TRAINING_SEEDS:
    torch.manual_seed(training_seed)

    seed_model = nn.Sequential(
        nn.Linear(X_train_tensor.shape[1], 64),
        nn.ReLU(),
        nn.Dropout(p=0.1),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Dropout(p=0.1),
        nn.Linear(32, 1),
    )
    seed_optimizer = torch.optim.Adam(seed_model.parameters(), lr=1e-3)
    seed_loss_function = nn.MSELoss()
    seed_best_loss = float("inf")
    seed_best_state = None
    seed_best_epoch = 0
    seed_patience_counter = 0

    for seed_epoch in range(1, NUM_EPOCHS + 1):
        seed_model.train()
        seed_optimizer.zero_grad()
        seed_train_loss = seed_loss_function(
            seed_model(X_train_tensor), y_train_tensor
        )
        seed_train_loss.backward()
        seed_optimizer.step()

        seed_model.eval()
        with torch.no_grad():
            seed_val_loss = seed_loss_function(
                seed_model(X_val_tensor), y_val_tensor
            ).item()

        if seed_val_loss < seed_best_loss:
            seed_best_loss = seed_val_loss
            seed_best_epoch = seed_epoch
            seed_best_state = copy.deepcopy(seed_model.state_dict())
            seed_patience_counter = 0
        else:
            seed_patience_counter += 1

        if seed_patience_counter >= PATIENCE:
            break

    seed_model.load_state_dict(seed_best_state)
    seed_model_states[training_seed] = copy.deepcopy(seed_best_state)
    seed_model.eval()
    with torch.no_grad():
        seed_predictions_scaled = seed_model(X_val_tensor).cpu().numpy()

    seed_predictions_rating = target_scaler.inverse_transform(
        seed_predictions_scaled
    ).ravel()
    seed_errors = seed_predictions_rating - y_val.to_numpy()
    seed_validation_predictions[training_seed] = seed_predictions_rating.copy()
    seed_results.append({
        "Seed": training_seed,
        "Best epoch": seed_best_epoch,
        "Validation MAE": np.mean(np.abs(seed_errors)),
        "Validation RMSE": np.sqrt(np.mean(seed_errors ** 2)),
    })
    print(f"Completed training seed {training_seed}.")

seed_results_df = pd.DataFrame(seed_results)
print(seed_results_df.to_string(index=False))

for metric in ["Validation MAE", "Validation RMSE"]:
    metric_mean = seed_results_df[metric].mean()
    metric_std = seed_results_df[metric].std(ddof=1)
    print(f"MLP {metric}: {metric_mean:.2f} +/- {metric_std:.2f} rating points")

print("Standard deviations describe training-seed variability on one fixed split.")


Completed training seed 42.
Completed training seed 43.
Completed training seed 44.
 Seed  Best epoch  Validation MAE  Validation RMSE
   42         200      145.890124       184.176844
   43         200      145.843985       183.738093
   44         198      145.737655       183.517414
MLP Validation MAE: 145.82 +/- 0.08 rating points
MLP Validation RMSE: 183.81 +/- 0.34 rating points
Standard deviations describe training-seed variability on one fixed split.


In [ ]:
# import hashlib
# import json
# import platform
# import importlib.metadata
# import joblib

# OUTPUT_DIR = PROJECT_ROOT / "outputs" / "mlp_midterm"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# dataset_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
# config = {
#     "experiment": "Current aggregate-feature rating estimation pilot",
#     "dataset_sha256": dataset_hash,
#     "feature_columns": feature_columns,
#     "target_column": "target_rating",
#     "split_seed": RANDOM_SEED,
#     "split_counts": {"train": len(train_df), "validation": len(val_df), "test": len(test_df)},
#     "training_seeds": TRAINING_SEEDS,
#     "hidden_dimensions": [64, 32],
#     "activation": "ReLU",
#     "dropout": 0.1,
#     "optimizer": "Adam",
#     "learning_rate": 0.001,
#     "max_epochs": NUM_EPOCHS,
#     "early_stopping_patience": PATIENCE,
#     "selection_metric": "Validation MSE on standardized targets",
#     "ridge_alpha": 1.0,
#     "imputation": "Infinite inputs replaced by NaN; training-median imputation",
#     "scaling": "Feature and target StandardScaler fitted on training data only",
#     "test_policy": "Evaluate all predetermined seeds; no seed selection or tuning by test results",
# }
# config_path = OUTPUT_DIR / "config.json"
# if config_path.exists():
#     if json.loads(config_path.read_text(encoding="utf-8")) != config:
#         raise RuntimeError("The frozen configuration differs. Use a separate experiment directory.")
# else:
#     config_path.write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")

# split_table = pd.concat([
#     train_df[["username"]].assign(split="train"),
#     val_df[["username"]].assign(split="validation"),
#     test_df[["username"]].assign(split="test"),
# ], ignore_index=True)
# assert split_table["username"].is_unique
# assert set(split_table["username"]) == set(df["username"])
# split_table.to_csv(OUTPUT_DIR / "player_splits.csv", index=False)

# joblib.dump({
#     "feature_columns": feature_columns,
#     "feature_imputer": feature_imputer,
#     "feature_scaler": feature_scaler,
#     "target_scaler": target_scaler,
# }, OUTPUT_DIR / "preprocessing.joblib")
# joblib.dump(ridge_model, OUTPUT_DIR / "ridge_model.joblib")
# for training_seed, state in seed_model_states.items():
#     torch.save(state, OUTPUT_DIR / f"mlp_seed_{training_seed}.pt")

# metric_rows = []
# prediction_tables = []

# def record_predictions(model_name, training_seed, split_name, players, targets, predictions):
#     errors = predictions - targets
#     metric_rows.append({
#         "model": model_name,
#         "seed": training_seed,
#         "split": split_name,
#         "mae": float(np.mean(np.abs(errors))),
#         "rmse": float(np.sqrt(np.mean(errors ** 2))),
#     })
#     prediction_tables.append(pd.DataFrame({
#         "username": players.to_numpy(),
#         "split": split_name,
#         "model": model_name,
#         "seed": training_seed,
#         "target_rating": targets,
#         "predicted_rating": predictions,
#         "error": errors,
#         "absolute_error": np.abs(errors),
#     }))

# for split_name, split_df, scaled_inputs, tensor_inputs in [
#     ("validation", val_df, X_val_scaled, X_val_tensor),
#     ("test", test_df, X_test_scaled, X_test_tensor),
# ]:
#     targets = split_df["target_rating"].to_numpy()
#     players = split_df["username"]
#     record_predictions("Training mean", None, split_name, players, targets,
#                        np.full(len(targets), float(y_train.mean())))
#     record_predictions("Ridge", None, split_name, players, targets,
#                        ridge_model.predict(scaled_inputs))
#     for training_seed in TRAINING_SEEDS:
#         evaluation_model = nn.Sequential(
#             nn.Linear(X_train_tensor.shape[1], 64), nn.ReLU(), nn.Dropout(p=0.1),
#             nn.Linear(64, 32), nn.ReLU(), nn.Dropout(p=0.1), nn.Linear(32, 1),
#         )
#         evaluation_model.load_state_dict(seed_model_states[training_seed])
#         evaluation_model.eval()
#         with torch.no_grad():
#             standardized_predictions = evaluation_model(tensor_inputs).cpu().numpy()
#         rating_predictions = target_scaler.inverse_transform(standardized_predictions).ravel()
#         record_predictions("MLP", training_seed, split_name, players, targets, rating_predictions)

# metrics_by_run = pd.DataFrame(metric_rows)
# predictions_by_player = pd.concat(prediction_tables, ignore_index=True)
# assert np.isfinite(predictions_by_player[["target_rating", "predicted_rating"]]).all().all()
# assert not predictions_by_player.duplicated(["username", "split", "model", "seed"]).any()
# metrics_by_run.to_csv(OUTPUT_DIR / "metrics_by_run.csv", index=False)
# predictions_by_player.to_csv(OUTPUT_DIR / "predictions_by_player.csv", index=False)

# summary_rows = []
# for (split_name, model_name), group in metrics_by_run.groupby(["split", "model"], sort=False):
#     summary_rows.append({
#         "split": split_name, "model": model_name, "runs": len(group),
#         "mae_mean": group["mae"].mean(),
#         "mae_std": group["mae"].std(ddof=1) if len(group) > 1 else 0.0,
#         "rmse_mean": group["rmse"].mean(),
#         "rmse_std": group["rmse"].std(ddof=1) if len(group) > 1 else 0.0,
#     })
# metrics_summary = pd.DataFrame(summary_rows)
# metrics_summary.to_csv(OUTPUT_DIR / "metrics_summary.csv", index=False)
# seed_results_df.to_csv(OUTPUT_DIR / "training_runs.csv", index=False)

# versions = {"python": platform.python_version()}
# for package in ["numpy", "pandas", "scikit-learn", "torch", "joblib"]:
#     versions[package] = importlib.metadata.version(package)
# (OUTPUT_DIR / "environment.json").write_text(json.dumps(versions, indent=2) + "\n", encoding="utf-8")

# table_lines = ["| Split | Model | MAE (rating points) | RMSE (rating points) |", "|---|---|---:|---:|"]
# for row in summary_rows:
#     mae_text = f"{row['mae_mean']:.2f}"
#     rmse_text = f"{row['rmse_mean']:.2f}"
#     if row["runs"] > 1:
#         mae_text += f" +/- {row['mae_std']:.2f}"
#         rmse_text += f" +/- {row['rmse_std']:.2f}"
#     table_lines.append(f"| {row['split']} | {row['model']} | {mae_text} | {rmse_text} |")

# brief = """# MLP baseline: midterm brief

# ## Baseline objective
# How much rating information can be recovered from a player's own aggregate features, without using graph edges?

# ## Method
# The pilot uses 7,571 players and 14 numeric features. Player IDs and target ratings are excluded from inputs. A fixed random player split contains 5,299 training, 1,136 validation, and 1,136 test players. Infinite feature values are treated as missing and filled with training-set medians. Feature and target scalers are fitted only on the training set.

# The MLP has hidden dimensions 64 and 32, ReLU activations, dropout 0.1, and a linear output. It uses MSE loss and Adam with learning rate 0.001, up to 200 epochs, and early-stopping patience 20. Each run restores the checkpoint with minimum validation MSE. Training seeds are 42, 43, and 44. Baselines are the training-set mean rating and Ridge regression with alpha 1.0.

# ## Results
# """ + "\n".join(table_lines) + """

# MLP values are mean +/- sample standard deviation across three independent training seeds on one fixed player split. They describe training variability, not uncertainty across players or data splits. Baselines are deterministic and evaluated once.

# ## Interpretation
# On the held-out test set, MLP has a mean RMSE of 186.38 rating points, compared with 193.48 for Ridge and 296.53 for the training-mean predictor. This is a 3.67% RMSE reduction relative to Ridge and a 37.15% reduction relative to the mean predictor. The pilot shows predictive information in player features and a modest additional benefit from the nonlinear model. The comparison does not establish a graph-model advantage or an improvement over Glicko-2. A fair GNN comparison still requires matching players, labels, available information, preprocessing, and model-selection rules.

# ## Limitations
# This pilot estimates the rating recorded for a player's latest sampled game from aggregate features. It is not yet an experiment using the first k games to predict a future converged rating. Some features include lifetime statistics and the sampled windows are not aligned to a common prediction cutoff. Random player splitting does not address these temporal issues. The rating is a proxy target rather than verified true skill. Sampling players with analyzed games can introduce selection bias. Graph models and Glicko-2 have not been evaluated in this notebook.

# ## Next steps
# Agree on a shared temporal input window and label definition, reproduce the baselines on that dataset, compare against teammates' graph models, and join spectral subgroup labels to saved player predictions. Complete the assigned literature contributions for the final paper.

# ## Short presentation script
# I implemented a non-graph MLP baseline to measure how much rating information is available in player features alone. We use training-only preprocessing, a shared player split, and validation-based checkpoint selection. We compare against a constant mean predictor and Ridge regression, and repeat MLP training with three seeds. The current results are a pilot on aggregate features; our next step is to align the temporal task and compare the graph models under the same protocol.
# """
# (OUTPUT_DIR / "midterm_brief.md").write_text(brief, encoding="utf-8")

# handoff = """# Reproducing and comparing the MLP pilot

# - Run the notebook from top to bottom in the repository Python environment.
# - config.json freezes data identity, feature order, architecture, split seed, training seeds, and evaluation rules before test evaluation.
# - player_splits.csv maps every player to train, validation, or test. Join by username; do not rely on CSV row numbers.
# - predictions_by_player.csv contains separate predictions for each model, split, and MLP seed. Do not treat three seed predictions as three independent players.
# - metrics_by_run.csv contains individual-run metrics. metrics_summary.csv contains model-level summaries.
# - preprocessing.joblib stores training-fitted preprocessing and the feature order; ridge_model.joblib stores Ridge; mlp_seed_*.pt store MLP state dictionaries.
# - environment.json records runtime package versions. training_runs.csv records selected epochs and validation metrics.
# - The notebook has already evaluated the frozen pilot on the held-out test set. Do not tune this pilot using its test results.
# - This split is proposed for team reuse; agreement with teammates is still required. Graph construction and temporal leakage require separate checks.
# - Checkpoints reflect current aggregate-feature data and must be regenerated when the task or feature definitions change.
# """
# (OUTPUT_DIR / "README.md").write_text(handoff, encoding="utf-8")
# print(metrics_summary.to_string(index=False))
# print("Saved midterm handoff to:", OUTPUT_DIR)


     split         model  runs   mae_mean  mae_std  rmse_mean  rmse_std
validation Training mean     1 240.198448 0.000000 303.895998  0.000000
validation         Ridge     1 153.122850 0.000000 193.831321  0.000000
validation           MLP     3 145.823921 0.078189 183.810784  0.335671
      test Training mean     1 236.451074 0.000000 296.534974  0.000000
      test         Ridge     1 155.651542 0.000000 193.484263  0.000000
      test           MLP     3 149.513466 1.241516 186.376330  1.259357
Saved midterm handoff to: C:\Users\Garrus\Documents\GitHub\GraphML-Lichess\outputs\mlp_midterm
